In [1]:
import pandas as pd
from IPython import display

apartments_df = pd.read_csv("data/processed_apartments_data_set.csv")
display(apartments_df.head(10))

,last_price,total_area,first_day_exposition,rooms,ceiling_height,floors_total,living_area,floor,studio,kitchen_area,balcony,locality_name,price_per_sqm
0,7312500,108.00,2024-05-15,3,2.70,16,51.000000,8,False,25.00,0,Kyiv,67708.333333
1,1884375,40.40,2024-08-14,1,2.65,11,18.600000,1,False,11.00,2,Brovary,46642.945545
2,2922750,56.00,2023-11-06,2,2.65,5,34.300000,4,False,8.30,0,Kyiv,52191.964286
3,36506250,159.00,2024-03-19,3,2.65,14,90.340909,9,False,9.10,0,Kyiv,229599.056604
4,5625000,100.00,2024-06-12,2,3.03,14,32.000000,13,False,41.00,0,Kyiv,56250.000000
5,1625625,30.40,2024-10-08,1,2.65,12,14.400000,5,False,9.10,0,Boyarka,53474.506579
6,2081250,37.30,2024-03-24,1,2.65,26,10.600000,6,False,14.40,1,Hostomel,55797.587131
7,4452188,71.60,2024-11-16,2,2.65,24,40.681818,22,False,18.90,2,Kyiv,62181.396648
8,1631250,33.16,2024-03-09,1,2.65,27,15.430000,26,False,8.81,0,Bucha,49193.305187
9,3037500,61.00,2024-03-10,3,2.50,9,43.600000,7,False,6.50,2,Kyiv,49795.081967


In [4]:
apartments_df.groupby("locality_name").size().sort_values(ascending=False)

locality_name
Kyiv             15651
Hostomel          1025
Bucha             1003
Brovary           1001
Boyarka           1000
Irpin              993
Vyshneve           991
Boryspil           986
Borshchahivka      963
dtype: int64

In [7]:
apartments_df[apartments_df["studio"] == True].groupby("locality_name").size().sort_values(ascending=False)

locality_name
Kyiv             85
Hostomel         12
Bucha            11
Brovary           8
Boyarka           8
Vyshneve          7
Borshchahivka     5
Boryspil          5
Irpin             5
dtype: int64

In [10]:
apartments_df.pivot_table(values="last_price", index="locality_name", columns="rooms", aggfunc="count", fill_value=0)

rooms,0,1,2,3,4,5,6,7,8,9,10,11,12,14,15,16,19
locality_name,,,,,,,,,,,,,,,,,
Borshchahivka,8,360,327,228,32,5,1,2,0,0,0,0,0,0,0,0,0
Boryspil,8,397,369,186,23,2,1,0,0,0,0,0,0,0,0,0,0
Boyarka,12,411,329,220,24,3,0,0,1,0,0,0,0,0,0,0,0
Brovary,13,382,360,202,41,3,0,0,0,0,0,0,0,0,0,0,0
Bucha,11,406,344,209,28,4,1,0,0,0,0,0,0,0,0,0,0
Hostomel,17,398,362,222,19,7,0,0,0,0,0,0,0,0,0,0,0
Irpin,6,358,398,199,24,7,1,0,0,0,0,0,0,0,0,0,0
Kyiv,109,4912,5082,4107,966,290,98,57,11,8,3,2,1,2,1,1,1
Vyshneve,10,391,342,222,20,4,2,0,0,0,0,0,0,0,0,0,0


In [13]:
numeric_cols = ["total_area","rooms","ceiling_height","floors_total",
                "living_area","floor","kitchen_area","balcony",
                "last_price","price_per_sqm"]

for col in numeric_cols:
    Q1 = apartments_df[col].quantile(0.25)
    Q3 = apartments_df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5*IQR
    upper = Q3 + 1.5*IQR
    outliers = apartments_df[(apartments_df[col] < lower) | (apartments_df[col] > upper)]
    print(col, Q1, Q3, lower, upper, len(outliers))

total_area 40.0 69.8 -4.699999999999996 114.5 1238
rooms 1.0 3.0 -2.0 6.0 90
ceiling_height 2.6 2.7 2.45 2.8500000000000005 2956
floors_total 5.0 16.0 -11.5 32.5 32
living_area 19.0 42.1 -15.650000000000006 76.75 885
floor 2.0 8.0 -7.0 17.0 906
kitchen_area 7.3 11.42 1.12 17.6 1551
balcony 0.0 1.0 -1.5 2.5 568
last_price 1912500.0 3824438.0 -955407.0 6692345.0 2049
price_per_sqm 43066.40625 64240.94707520892 11304.595012186623 96002.75831302229 1091


In [14]:
print(apartments_df["ceiling_height"].describe())
print(apartments_df["ceiling_height"].value_counts().sort_index())

count    23613.000000
mean         2.724581
std          0.990085
min          1.000000
25%          2.600000
50%          2.650000
75%          2.700000
max        100.000000
Name: ceiling_height, dtype: float64
ceiling_height
1.00       1
1.20       1
1.75       1
2.00      11
2.20       1
          ..
26.00      1
27.00      8
27.50      1
32.00      2
100.00     1
Name: count, Length: 183, dtype: int64


In [16]:
apartments_df = apartments_df[apartments_df["ceiling_height"].between(2.0, 4.5)]
Q1 = apartments_df["rooms"].quantile(0.25)
Q3 = apartments_df["rooms"].quantile(0.75)
IQR = Q3 - Q1
apartments_df = apartments_df[apartments_df["rooms"] <= Q3 + 1.5*IQR]

print(apartments_df.shape)

(23479, 13)


In [24]:
apartments_df.pivot_table(values="last_price", index="locality_name", columns="rooms", aggfunc="count", fill_value=0)


rooms,0,1,2,3,4,5,6
locality_name,,,,,,,
Borshchahivka,8,360,326,228,32,5,1
Boryspil,8,396,368,185,23,2,1
Boyarka,12,409,327,220,24,3,0
Brovary,13,382,360,202,40,3,0
Bucha,11,406,344,209,28,4,1
Hostomel,16,397,362,222,19,7,0
Irpin,6,358,395,199,24,7,1
Kyiv,108,4904,5072,4102,965,286,97
Vyshneve,10,391,342,222,20,4,2


In [25]:
apartments_df[apartments_df["locality_name"] == "Irpin"]["balcony"].mean()

np.float64(0.5909090909090909)

In [26]:
apartments_df[apartments_df["locality_name"] == "Irpin"].agg(
    balcony_mean=("balcony", "mean")
)

,balcony
balcony_mean,0.590909


In [27]:
display(apartments_df)

,last_price,total_area,first_day_exposition,rooms,ceiling_height,floors_total,living_area,floor,studio,kitchen_area,balcony,locality_name,price_per_sqm
0,7312500,108.00,2024-05-15,3,2.70,16,51.000000,8,False,25.00,0,Kyiv,67708.333333
1,1884375,40.40,2024-08-14,1,2.65,11,18.600000,1,False,11.00,2,Brovary,46642.945545
2,2922750,56.00,2023-11-06,2,2.65,5,34.300000,4,False,8.30,0,Kyiv,52191.964286
3,36506250,159.00,2024-03-19,3,2.65,14,90.340909,9,False,9.10,0,Kyiv,229599.056604
4,5625000,100.00,2024-06-12,2,3.03,14,32.000000,13,False,41.00,0,Kyiv,56250.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
23608,5456250,133.81,2024-06-23,3,3.70,5,73.300000,3,False,13.83,0,Kyiv,40776.100441
23609,1743750,59.00,2023-11-22,3,2.65,5,38.000000,4,False,8.50,0,Borshchahivka,29555.084746
23610,1406250,56.70,2024-07-22,2,2.65,3,29.700000,1,False,9.10,0,Boryspil,24801.587302
23611,6454688,76.75,2023-09-23,2,3.00,17,43.607955,12,False,23.30,2,Kyiv,84100.169381


In [30]:
apartments_df[apartments_df["locality_name"] == "Kyiv"].pivot_table(
    values="price_per_sqm",
    index="rooms",
    aggfunc="mean"
)

,price_per_sqm
rooms,
0,69546.588404
1,65161.739083
2,63428.852111
3,62708.432955
4,67523.861697
5,74002.656191
6,80803.950918


In [31]:
apartments_df[apartments_df["locality_name"] == "Kyiv"].pivot_table(
    values="price_per_sqm",
    index="ceiling_height",
    aggfunc="mean"
)

,price_per_sqm
ceiling_height,
2.00,64355.698923
2.30,50575.657895
2.40,57622.552581
2.45,47901.282098
2.46,66736.641221
...,...
4.30,71072.635135
4.37,55114.311164
4.40,60900.297619
